# 01 — Dataset overview

What the 2026-09-09 scrape of [climbing-history.org](https://climbing-history.org)
holds, and a raw-vs-cleaned check of each known defect. The three defects and
their history are documented in `clean.py` and HANDOFF.md; this notebook
verifies them against the committed files rather than asserting them.

Conventions: printed output stays small (aggregates and example rows, never a
whole file); everything here reads the committed `data/` and writes nothing.

In [1]:
import pandas as pd

from sportgradehistory import config
from sportgradehistory.clean import clean_detail_frame, is_known_type, unknown_types
from sportgradehistory.grades import french_ordinal

raw = pd.read_csv(config.RAW_DETAIL_CSV)
clean = pd.read_csv(config.CLEAN_DETAIL_CSV)
index = pd.read_csv(config.CLEAN_INDEX_CSV)
sport = pd.read_csv(config.SPORT_CSV)
boulders = pd.read_csv(config.BOULDERS_CSV)

for name, df in [("raw detail", raw), ("clean detail", clean), ("index", index),
                 ("sport_routes", sport), ("boulders", boulders)]:
    print(f"{name:<14} {df.shape}")

raw detail     (7669, 11)
clean detail   (7669, 11)
index          (8114, 8)
sport_routes   (2027, 17)
boulders       (3098, 16)


## What was scraped

One row per climb page with at least one recorded successful ascent, every
climb type kept — the boulder rows are the control group for the sport
analysis, and both datasets come out of this one snapshot.

In [2]:
import json

manifest = json.loads(config.SCRAPE_MANIFEST.read_text())
for section in ("index", "details"):
    m = manifest[section]
    print(f"{section:<8} {m['started_utc'][:10]}  rows={m['rows']:,}  "
          f"elapsed={m['elapsed']}  errors={m['fetch_errors']}")

index    2026-09-09  rows=8,114  elapsed=0:09:00  errors=0
details  2026-09-09  rows=7,669  elapsed=1:52:00  errors=0


In [3]:
print(clean["climb_type"].value_counts(dropna=False).to_string())

climb_type
Boulder problem                      3077
Trad climb                           2069
Sport route                          1976
Multi-pitch                           296
Deep water solo                        87
Aid Climb                              44
Boulder problem (indoor)               27
(approx) Boulder problem               21
Traverse                               16
Alpine                                 16
(approx) Trad climb                    15
NaN                                    13
(approx) Multi-pitch                    6
(approx) Sport route                    2
(approx) Traverse                       2
(approx) Boulder problem (indoor)       1
(approx) Aid Climb                      1


## Defect checks, raw vs cleaned

The boulder repo's scrape had three defects. The current scraper heads 1 and 2
off at the source; 3 still occurs and is repaired by `clean_detail_frame`.
Check all three here instead of trusting the story.

In [4]:
# Defect 1: trailing navigation text ("More Change Log Notes Log Threads").
nav = r"Change Log|Notes Log|More Notes|\bThreads\b"
for frame, label in ((raw, "raw"), (clean, "clean")):
    n = sum(
        frame[c].dropna().astype(str).str.contains(nav, regex=True).sum()
        for c in ("grade", "climb_type", "location", "first_climber")
    )
    print(f"defect 1 ({label:<5}): {n} affected cells")

defect 1 (raw  ): 0 affected cells
defect 1 (clean): 0 affected cells


In [5]:
# Defect 2: location fused into climb_type ("Sport route in Chee Dale").
for frame, label in ((raw, "raw"), (clean, "clean")):
    t = frame["climb_type"].dropna().astype(str)
    n = t.str.contains(r"\s(?:in|on|at)\s").sum()
    print(f"defect 2 ({label:<5}): {n} fused rows")

defect 2 (raw  ): 0 fused rows
defect 2 (clean): 0 fused rows


In [6]:
# Defect 3: column shift on ungraded routes. Present in this snapshot.
shifted = raw[raw["grade"].notna() & ~raw["climb_type"].map(is_known_type)]
print(f"defect 3 (raw): {len(shifted)} shifted rows\n")
cols = ["climb_id", "climb_name", "grade", "climb_type"]
print(shifted[cols].to_string(index=False))
print("\nafter cleaning, the same climb_ids:")
repaired = clean[clean["climb_id"].isin(shifted["climb_id"])]
print(repaired[["climb_id", "climb_name", "grade", "climb_type", "location"]]
      .to_string(index=False))

defect 3 (raw): 9 shifted rows

 climb_id                   climb_name      grade                    climb_type
     2627              Southeast Ridge        Mt.                       Everest
     2641                   West Ridge  Annapurna                            II
     2772              South-west face        Mt.                       Everest
     3098                    West Face      Siula                        Grande
     3724                 Mazeno Ridge      Nanga                        Parbat
     3732                Rakhiot Flank      Nanga                        Parbat
     3765 Edge of Entropy (West Ridge) Gasherbrum                           III
     5280             Compressor Route      Cerro                         Torre
     5873      Substance of Everything         8B Boulder problem The Frontline

after cleaning, the same climb_ids:
 climb_id                   climb_name grade climb_type                         location
     2627              Southeast Ridge   N

Eight alpine routes come back correctly (`grade`/`climb_type` cleared, the
location rejoined). The ninth, climb 5873, is the known cost documented in
HANDOFF.md: its heading fuses the location with no preposition, so the repair
cannot tell it from a column shift and one 8B boulder drops out of the control
group. The type vocabulary check below is the alarm that would catch a new
type being misread the same way — it should list nothing but that row's
debris.

In [7]:
print(unknown_types(raw).to_string())

climb_type
Everest                          2
Parbat                           2
II                               1
Grande                           1
III                              1
Torre                            1
Boulder problem The Frontline    1


## The 63 rows with ascents but no first-ascent fields

Diagnosed in HANDOFF.md: when a climb's chronologically first table entry is
an *unsuccessful* attempt, the site server-renders only that leading DNF row,
so the scraper sees no successful-ascent row at all. `num_ascents` is still
correct (it comes from the page text). The affected set skews toward
significant routes, because those attract recorded pre-FA attempts.

In [8]:
gap = clean[(clean["num_ascents"] > 0) & clean["first_climber"].isna()]
print(f"{len(gap)} rows, by type:")
print(gap["climb_type"].value_counts(dropna=False).to_string())
notable = gap[gap["num_ascents"] >= 5]
print(f"\nthe {len(notable)} with 5+ ascents:")
print(notable[["climb_id", "climb_name", "grade", "num_ascents"]]
      .sort_values("num_ascents", ascending=False).to_string(index=False))

63 rows, by type:
climb_type
Sport route             18
Trad climb              17
Boulder problem         15
Multi-pitch              7
Aid Climb                2
Traverse                 1
Deep water solo          1
NaN                      1
(approx) Multi-pitch     1

the 20 with 5+ ascents:
 climb_id          climb_name  grade  num_ascents
     1146       Off the Wagon    8B+           23
      174             Voyager     8B           18
      178              Cypher     8B           16
      519    Punks in the Gym    8b+           12
      789        Chilam Balam    9a+           11
     1205           Es Pontas    9a+           10
      144          The Island    8B+            8
     1240          Impact Day     E8            8
     1373     Cenotaph Corner     E1            8
      419        Little Women    8B+            8
     5459         Butterballs E4/6c+            7
       56            A Muerte    8c+            7
      391     Northern Lights     9a            6
   

Punks in the Gym and Chilam Balam sit in this set — both matter to the
milestone story, and `milestones.py` handles them explicitly (a cited
supplement for Punks; Chilam Balam is a disputed claim either way).

## The sport dataset at a glance

2,027 rows: every `Sport route` (all French-graded) plus the 49 French-graded
multi-pitches, flagged. The exact filter and its exclusions are in HANDOFF.md.

In [9]:
print("is_multipitch:", sport["is_multipitch"].value_counts().to_dict())
print("status of FA date:", sport["first_ascent"].notna().sum(), "dated,",
      sport["first_ascent"].isna().sum(), "unknown\n")
counts = sport["grade_clean"].value_counts()
counts = counts.reindex([g for g in counts.index], fill_value=0)
order = sorted(counts.index, key=french_ordinal)
print("routes per grade (8a+ and up):")
top = {g: int(counts[g]) for g in order if french_ordinal(g) >= french_ordinal("8a+")}
print(top)

is_multipitch: {False: 1978, True: 49}
status of FA date: 1938 dated, 89 unknown

routes per grade (8a+ and up):
{'8a+': 112, '8b': 223, '8b+': 224, '8c': 302, '8c+': 234, '9a': 322, '9a+': 175, '9b': 76, '9b+': 11, '9c': 5}


In [10]:
print("first ascent styles on sport routes:")
print(sport["first_style"].value_counts(dropna=False).head(8).to_string())

first ascent styles on sport routes:
first_style
Lead | worked               1739
Lead | onsight               128
Lead | flash                  60
Solo | worked                 22
NaN                           18
Alternate Leads | worked      17
Lead                          15
Top Rope | worked             10


## The control group

3,098 outdoor boulder problems from the same snapshot, Font-ordered. The
sister repo ([bouldering-grade-history](https://github.com/joeykang21-arch/bouldering-grade-history))
analyses these properly; here they exist so the sport-vs-boulder progression
comparison uses one scrape date for both.

In [11]:
top_boulders = boulders[boulders["grade_order"] >= boulders.loc[
    boulders["grade_clean"] == "8A", "grade_order"].iloc[0]]
print("boulders 8A and up, per grade:")
print(top_boulders["grade_clean"].value_counts().sort_index().to_string())

boulders 8A and up, per grade:
grade_clean
8A     286
8A+    324
8B     879
8B+    682
8C     353
8C+     71
9A      14
9A+      1
